<a href="https://colab.research.google.com/github/LukeXboy/Healthcare-Readmission-Data-Science-Project/blob/main/01_data_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Healthcare Readmission Prediction — Day 1: Dataset Audit

**Question:** Can we predict a recorded 30-day readmission for an inpatient encounter involving diabetes using data available by discharge?

**Source:** [UCI Diabetes 130-US Hospitals, 1999–2008](https://archive.ics.uci.edu/dataset/296/diabetes%2B130-us%2Bhospitals%2Bfor%2Byears%2B1999-2008), DOI 10.24432/C5230J (CC BY 4.0). Retrospective historical US data; not a deployable clinical system.

**Today:** Explore the raw data without modifying it, and report observations. Don't fit a model or drop missing features yet.


In [3]:
from pathlib import Path
import pandas as pd


df = pd.read_excel('/content/diabetic_data.csv.xlsx', na_values=['?'])
print('Loaded dataset:', '/content/diabetic_data.csv.xlsx')
print('Shape:', df.shape)
display(df.head())


Loaded dataset: /content/diabetic_data.csv.xlsx
Shape: (101766, 50)


,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,...,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,2278392,8222157,Caucasian,Female,[0-10),NaN,6,25,1,1,...,No,No,No,No,No,No,No,No,No,NO
1,149190,55629189,Caucasian,Female,[10-20),NaN,1,1,7,3,...,No,Up,No,No,No,No,No,Ch,Yes,>30
2,64410,86047875,AfricanAmerican,Female,[20-30),NaN,1,1,7,2,...,No,No,No,No,No,No,No,No,Yes,NO
3,500364,82442376,Caucasian,Male,[30-40),NaN,1,1,7,2,...,No,Up,No,No,No,No,No,Ch,Yes,NO
4,16680,42519267,Caucasian,Male,[40-50),NaN,1,1,7,1,...,No,Steady,No,No,No,No,No,Ch,Yes,NO


## Part 1 — Data grain and missing values


In [4]:
print('Number of rows and columns:', df.shape)
print('Column names:', df.columns)
print('Number of distinct encounter_id values:', df['encounter_id'].nunique())
print('Number of distinct patient_nbr values:', df['patient_nbr'].nunique())
print('10 columns with the highest percentage of missing values:', df.isna().mean().sort_values(ascending=False).head(10))
print('Raw readmitted class counts and percentages:', df['readmitted'].value_counts(normalize=True))

Number of rows and columns: (101766, 50)
Column names: Index(['encounter_id', 'patient_nbr', 'race', 'gender', 'age', 'weight',
       'admission_type_id', 'discharge_disposition_id', 'admission_source_id',
       'time_in_hospital', 'payer_code', 'medical_specialty',
       'num_lab_procedures', 'num_procedures', 'num_medications',
       'number_outpatient', 'number_emergency', 'number_inpatient', 'diag_1',
       'diag_2', 'diag_3', 'number_diagnoses', 'max_glu_serum', 'A1Cresult',
       'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide',
       'glimepiride', 'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide',
       'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone',
       'tolazamide', 'examide', 'citoglipton', 'insulin',
       'glyburide-metformin', 'glipizide-metformin',
       'glimepiride-pioglitazone', 'metformin-rosiglitazone',
       'metformin-pioglitazone', 'change', 'diabetesMed', 'readmitted'],
      dtype='object')
Number of dist

##Part 2 - Train/Test Split





In [5]:
from sklearn.model_selection import GroupShuffleSplit
#using groupshufflesplit to prevent same patients appearing in both sets
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(
        df,
        groups=df['patient_nbr']
    )
)

train_df = df.iloc[train_idx].copy()
test_df = df.iloc[test_idx].copy()

In [7]:
train_patients = set(train_df['patient_nbr'])
test_patients = set(test_df['patient_nbr'])

overlap = train_patients.intersection(test_patients)

print("Training encounters:", len(train_df))
print("Test encounters:", len(test_df))
print("Training patients:", len(train_patients))
print("Test patients:", len(test_patients))
print("Overlapping patients:", len(overlap))

Training encounters: 81613
Test encounters: 20153
Training patients: 57214
Test patients: 14304
Overlapping patients: 0


In [8]:
#check class distribution in train and test set
print("Training distribution:")
print(
    train_df['readmitted'].value_counts(
        normalize=True
    )
)

print("\nTest distribution:")
print(
    test_df['readmitted'].value_counts(
        normalize=True
    )
)

Training distribution:
readmitted
NO     0.537292
>30    0.349907
<30    0.112801
Name: proportion, dtype: float64

Test distribution:
readmitted
NO     0.546519
>30    0.346747
<30    0.106733
Name: proportion, dtype: float64


In [9]:
#make binary target variable: readmitted_30d
train_df['readmitted_30d'] = (train_df['readmitted'] == '<30').astype(int)
test_df['readmitted_30d'] = (test_df['readmitted'] == '<30').astype(int)
#check percentage of positive and negative class in each set
print("Training distribution:")
print(
    train_df['readmitted_30d'].value_counts(
        normalize=True
    )
)

print("\nTest distribution:")
print(
    test_df['readmitted_30d'].value_counts(
        normalize=True
    )
)

Training distribution:
readmitted_30d
0    0.887199
1    0.112801
Name: proportion, dtype: float64

Test distribution:
readmitted_30d
0    0.893267
1    0.106733
Name: proportion, dtype: float64


In both train and test dataset, the class distribution is very imbalanced (about 11% are postive class). Therefore, later when we evaluate our model performance, we need to use PR-AUC, ROC-AUC, precision, recall, and F1 rather than relying on accuracy alone.